In [9]:
import os
import argparse
import torch
import torchvision
import pandas as pd
from tqdm import tqdm
from pytorch_fid import fid_score
from torchvision.utils import save_image

In [10]:
#choose cpu/gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [11]:
#load model.ipynb file
%run U-Net.ipynb

In [12]:
#Karras sample
@ torch.no_grad()
def get_sample_karras(steps, sigma_min=0.002, sigma_max=80, rho=7, device=device):
    ramp = torch.linspace(0, 1, steps, device=device)
    min_inv = sigma_min ** (1 / rho)
    max_inv = sigma_max ** (1 / rho)
    sigmas = (max_inv + ramp * (min_inv - max_inv)) ** rho
    sigmas = torch.cat([sigmas, torch.zeros_like(sigmas[:1])])
    return sigmas

In [13]:
#sample batch
@ torch.no_grad()
def sample_batch(model, batch_size, image_size, steps=50):
    #range of sigma
    sigma_min = 0.002
    sigma_max = 80
    sigmas = get_sample_karras(steps, sigma_min=0.002, sigma_max=80, rho=7, device=device)
    sigma_data = 0.5
    #random generate img
    x = torch.randn(batch_size, 3, image_size, image_size, device=device) * sigmas[0]
    #generate img without noise
    for i in range(len(sigmas) - 1):
        #choose values of sigma & sigma_next
        sigma_i = sigmas[i]
        sigma_next = sigmas[i+1]

        sigma_i_batch = sigma_i.expand(batch_size)
        sigma_i_reshaped = sigma_i.view(-1, 1, 1, 1)
        #EDM preconditioning
        c_skip = sigma_data ** 2 / (sigma_i_reshaped ** 2 + sigma_data ** 2)
        c_out = sigma_i_reshaped * sigma_data / torch.sqrt(sigma_i_reshaped ** 2 + sigma_data ** 2)
        c_in = 1 / torch.sqrt(sigma_i_reshaped ** 2 + sigma_data ** 2)
        #calculate derivative value
        pred = model(c_in * x, sigma_i_batch)
        x0_hat = c_skip * x + c_out * pred
        d_i = (x - x0_hat) / sigma_i_reshaped
        #Euler predict
        dt = sigma_next - sigma_i
        x_euler = x + dt * d_i
        
        if sigma_next == 0:
            x = x_euler
        else:
            sigma_next_batch = sigma_next.expand(batch_size)
            sigma_next_reshaped = sigma_next.view(-1, 1, 1, 1)
            #EDM preconditioning
            c_skip = sigma_data ** 2 / (sigma_next_reshaped ** 2 + sigma_data ** 2)
            c_out = sigma_next_reshaped * sigma_data / torch.sqrt(sigma_next_reshaped ** 2 + sigma_data ** 2)
            c_in = 1 / torch.sqrt(sigma_next_reshaped ** 2 + sigma_data ** 2)
            #calculate derivative value
            pred_next = model(c_in * x_euler, sigma_next_batch)
            x0_hat_next = c_skip * x_euler + c_out * pred_next
            #Heun correction
            d_prime = (x_euler - x0_hat_next) / sigma_next_reshaped
            x = x + dt * 0.5 * (d_i + d_prime)
    return x

In [14]:
#generate sample
@ torch.no_grad()
def generate_sample(model, save_dir, num_samples=5000, batch_size=64, image_size=32, use_tqdm=True):
    #create model dir
    if not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
    #update the bar
    total = 0
    pbar = tqdm(total=num_samples, disable=not use_tqdm)
    #generate images
    while total < num_samples:
        #calculate batch_size
        current_bs = min(batch_size, num_samples - total)
        #generate samples
        samples = sample_batch(model, current_bs, image_size)
        samples = samples.clamp(-1, 1)
        samples = (samples + 1) / 2
        samples = samples.clamp(0, 1)
        #save samples
        for i in range(current_bs):
            save_path = os.path.join(save_dir, f'{total+i:05d}.png')
            save_image(samples[i], save_path)
        total += current_bs
        pbar.update(current_bs)
    pbar.close()

In [15]:
#compute fid
def compute_fid(real_dir, fake_dir):
    fid = fid_score.calculate_fid_given_paths(
        [real_dir, fake_dir],
        batch_size=64,
        device=device,
        dims=2048
    )
    return fid